# 一、 資金配置方法與交易邏輯

配對交易 (Pairs Trading) 作為一種經典的統計套利 (Statistical Arbitrage) 策略，其核心不僅在於尋找高度相關的資產對，更在於**科學化的資金配置**與**嚴謹的進出場邏輯**。本專案建構了一套高頻滾動的實戰級資金與交易引擎。

## 1.1 多視窗滾動回測架構

股票之間的相關性 or 共整合關係具有隨時間漂移的**非定態 (Non-stationary) 特性**。為了解決統計參數過時的風險，本系統採用高頻滾動視窗進行動態配對篩選與交易：

*   **形成期 (Formation Period)**: $252$ 天（約一年），用於統計參數計算、聚類及共整合篩選。
*   **交易期 (Trading Period)**: $126$ 天（約半年），用於實際監控價差與信號交易。
*   **滾動步長 (Rolling Step)**: $21$ 天（約一個月），代表每個月系統會動態汰換並重新篩選配對組合。

### 💰 資金槽分流設計 (Slots Mechanism)
由於每一期交易期為 $126$ 天，而每隔 $21$ 天就會啟動一期新的交易，因此在任何一個交易日，系統最多會有 $\lfloor 126 / 21 \rfloor = 6$ 個不同時期篩選出來的配對組合在同時運行。為了防止資金過度重疊或交易穿透，系統將帳戶初始資金（$\$10,000$）等分為 **$6$ 個獨立的資金槽 (Slots)**，每個資金槽分配 $\approx \$1,666.67$。

當某一期交易期（$126$ 天）結束後，該槽的資金連同累計盈虧將完全釋放，並在下一次滾動步長點上重新作為新一期交易的基礎資金，實現**槽與槽之間資金完全獨立、槽內資金滾動複利**的安全設計。

## 1.2 等權重資金配置與避險部位計算

在單一資金槽內，針對該期篩選出的 $N$ 組最佳配對（$Top\ N$），策略採用**等權重 (Equal Weight)** 方式分配資金。每組配對分配到的資金為：
$$capital\\_per\\_pair = \\frac{capital\\_per\\_slot}{N}$$

### ⚖️ 基於 $\\beta$ (Hedge Ratio) 的市場中性部位
設選定配對為 $Y$ (Ticker_A) 與 $X$ (Ticker_B)，其避險比率為 $\\beta$（來自 OLS 滾動殘差回歸或 Engle-Granger 估計）。為了達到市場中性（避險權重平衡），兩檔股票的資金配置比例應嚴格對齊避險比率，即 $1 : |\\beta|$。

分配給股票 A 與股票 B 的資金價值（$v_A$ 與 $v_B$）計算如下：
$$v_A = capital\\_per\\_pair \\times \\frac{1.0}{1.0 + |\\beta|}, \\quad v_B = capital\\_per\\_pair \\times \\frac{|\\beta|}{1.0 + |\\beta|}$$

在交易信號觸發時（設當日價格分別為 $p_A, p_B$），兩標的的買賣股數（$shares\\_a, shares\\_b$）計算如下：

*   **賣空價差 (Short Spread)**：當價差過高，做空 A（$Y$）並做多 B（$X$）：
    $$shares\\_a = -\\frac{v_A}{p_A}, \\quad shares\\_b = \\frac{v_B}{p_B}$$
*   **買入價差 (Long Spread)**：當價差過低，做多 A（$Y$）並做空 B（$X$）：
    $$shares\\_a = \\frac{v_A}{p_A}, \\quad shares\\_b = -\\frac{v_B}{p_B}$$

## 1.3 交易摩擦成本與淨損益計算

實務交易中，忽略摩擦成本常導致回測績效嚴重高估。本平台引入了**雙邊摩擦成本**，將手續費率（Fee Rate）與滑價率（Slippage Rate）進行加總：
$$Friction = Fee\\_Rate + Slippage\\_Rate$$

### 📉 交易生命週期成本扣除邏輯：
1.  **進場手續費扣除 (Trade Entry Fee)**:
    在進場建倉當日，系統計算進場部位的總名目價值，並預先扣除手續費，計入該部位的交易成本，此時該配對的盈虧（Unrealized PnL）會因為摩擦成本而呈現負值：
    $$trade\\_entry\\_fee = (|shares\\_a| \\times p_A + |shares\\_b| \\times p_B) \\times Friction$$
2.  **出場手續費估計 (Exit Fee)**:
    在持倉期間，系統以每日最新的收盤價實時估計出場手續費：
    $$exit\\_fee\\_est = (|shares\\_a| \\times p_{A,t} + |shares\\_b| \\times p_{B,t}) \\times Friction$$
3.  **持倉未實現淨損益 (Net Unrealized PnL)**:
    $$Raw\\_Unrealized = shares\\_a \\times (p_{A,t} - p_{A,entry}) + shares\\_b \\times (p_{B,t} - p_{B,entry})$$
    $$Net\\_Unrealized\\_PnL = Raw\\_Unrealized - trade\\_entry\\_fee - exit\\_fee\\_est$$
4.  **出場已實現淨損益 (Net Realized PnL)**:
    當部位平倉時，以最終平倉價格計算最終出場手續費（$exit\\_fee$），並將扣除雙邊手續費後的淨損益計入該配對的 `realized_pnl`：
    $$PnL\\_Final = Raw\\_Unrealized\\_Final - trade\\_entry\\_fee - exit\\_fee$$

## 1.4 多層級風險防護機制

為了應對黑天鵝事件與配對關係徹底破裂（基本面分歧），本引擎實作了四道安全防線：

1.  **單筆個股停損 (Stop Loss, SL)**:
    當單筆配對的累計虧損達到該組配對分配資金的特定比例 $stop\\_loss\\_pct$（如 5% 或 15%）時，即：
    $$\\frac{-PnL}{capital\\_per\\_pair} \\ge stop\\_loss\\_pct$$
    系統將立即觸發強行平倉，以鎖定最大損失。
2.  **部位動態 Z-Score 停損 (DSZ)**:
    當 Z-Score 發生極端異常偏離，說明兩股價差完全失控。若 $|Z| > dynamic\\_stop\\_z$（如超過 3.0 或 5.0 個標準差），則執行動態技術性停損。
3.  **投資組合全域止損斷路器 (Portfolio Stop Loss, PSL)**:
    如果該期所有配對的每日累計損益達到整個資金槽的特定比例 $portfolio\\_stop\\_loss\\_pct$（如 10% 或 20%），則系統將**永久關閉該期剩餘的所有交易**，強行平倉所有部位，且該期不再進行 any 新開倉，起到全域斷路器作用。
4.  **重入限制與冷卻期 (Re-entry & Cooldown)**:
    當 `allow_reentry = False` 時，一旦某組配對觸發了停損，該配對在**該交易期內將被標記為 STOPPED，不再進行交易**。若為 True，則在價差冷卻回歸後允許重新建倉。

# 二、 策略形成期 (Formation Period) 深度解析

策略形成期統一設定為 **$252$ 個交易日**（約一年歷史價格）。其核心任務在於**數據清洗**、**特徵降維**、**無監督聚類**以及**統計共整合選股**，包含以下五大核心科學步驟：

### 1. 資料清理與對數 Z-Score 標準化
載入收盤價，進行對數轉換。對對數價格進行 Z-Score 標準化，徹底消除不同股價絕對水平的影響：
$$P_{norm} = \\frac{\\ln(P) - Mean(\\ln(P))}{Std(\\ln(P))}$$

### 2. 13維多維度特徵工程 (Feature Extraction)
為每支股票萃取 $13$ 個具有判別力的統計與動量指標，刻畫股票走勢指紋：
*   *動量特徵 (4維)*: 5日、21日、63日、126日 對數收益率。
*   *波動率特徵 (3維)*: 21日、63日滾動收益標準差，及全期價格標準差。
*   *自相關特徵 (3維)*: lag-1, lag-5, lag-21 收益自相關係數。
*   *統計矩與分形 (3維)*: 偏態 (Skewness)、峰態 (Kurtosis)、及 Hurst 指數近似值。

### 3. 非線性空間降維 (UMAP / PCA)
應用流形降維技術 **UMAP** (或 PCA) 將 13 維特徵投影至 $5$ 維嵌入空間，有效壓縮噪聲並保留流形幾何結構。

### 4. 無監督密度分群 (HDBSCAN)
在降維空間中，使用 HDBSCAN 進行層級密度分群（Min Cluster Size=3），自動決定群落結構並剔除標記為**噪音點 (Label = -1) 予以排除**，起到天生的防暴雷保護。

### 5. 同產業共整合檢定與定態篩選
僅對「同產業 $\\cap$ 同聚類群落」的股票進行 Engle-Granger 二階段檢定，估計 $\\beta$：
$$\\ln(p_{A,\\tau}) = \\alpha + \\beta \\ln(p_{B,\\tau}) + \\epsilon_{\\tau}$$
對殘差序列 $\\epsilon_{\\tau}$ 進行 ADF 定態檢定。僅挑選 **ADF p-value < 0.05** 且 Ornstein-Uhlenbeck 均值復歸半衰期介於 $2$ 至 $60$ 天的配對，按 ADF 統計量升序選出 $Top\\_N$。

# 三、 策略交易期 (Trading Period) 執行與風控解析

策略交易期統一設定為 **$126$ 個交易日**（約半年外推回測）。本專案在此期間以每日為單位，嚴格執行信號計算、等權重部位下單與多層級風險防護，確保回測的擬真度與實戰性：

### 📊 1. 每日信號監控與參數估計邏輯
在交易期的每一天 $t$，系統會依據設定計算最新價差 (Spread) 與 Z-Score 決定進出場：
*   **靜態參數外推模式 (Z-Window = 0)**：
    價差與 Z-Score 完全套用形成期所算出的 $\\beta_{form}$、Spread Mean 和 Spread Std 每日外推，公式為：
    $$Spread_t = \\ln(p_{A,t}) - \\beta_{form} \\ln(p_{B,t}), \quad Z_t = \\frac{Spread_t - Spread\\_Mean_{form}}{Spread\\_Std_{form}}$$
*   **動態滾動 OLS 模式 (Z-Window = W)**：
    每日取過去 $W$ 天（如 20 天）價格重新做 OLS 估計當日的避險比率 $\\beta_t$ 與 $\\alpha_t$，動態計算 Z-Score：
    $$Spread_t = \\ln(p_{A,t}) - \\alpha_t - \\beta_t \\ln(p_{B,t}), \quad Z_t = \\frac{Spread_t - Mean(\\epsilon_t)}{Std(\\epsilon_t)}$$
*   **波動度調節因子 (VolAdj) 抗震機制**：當市場異常波動，以過去 20 天滾動標準差自適應拓寬 Z 邊界分母，公式：
    $$\\sigma_{adj} = \\max\\left(\\sigma_{base},  \\sigma_{base} \\times \\frac{\\sigma_{roll20}}{\\sigma_{form}}\\right)$$

### 🛠️ 2. 部位等權重分配與雙邊手續費
在信號觸發進場（如 $|Z_t| > entry\\_z$）時，系統將配對資金 $cap$ 依 $\\beta$ 避險權重拆分配置：$v_A = cap \\times \\frac{1}{1+|\\beta|}, v_B = cap \\times \\frac{|\\beta|}{1+|\\beta|}$，實現多空金額完全市場中性。同時，在進場日預先扣除 $trade\\_entry\\_fee$，每日動態計算未實現淨盈虧（已扣預估出場費），並在平倉（正常回歸或停損）時扣除 $exit\\_fee$ 結算 realized PnL。

### 🛡️ 3. 三維硬性風險管理機制
*   **個股資金停損 (SL)**: 單組配對累計虧損達到分配資金的特定比例（如 5% 或 15%）時，即 $-PnL / cap \\ge stop\\_loss\\_pct$，強行平倉。若 `allow_reentry = False`，該組在該交易期內被永久鎖定，不再交易。
*   **部位技術停損 (DSZ)**: 當 Z-Score 發生極端異常偏離（如 $|Z_t| > 3.0$），代表共整合關係短暫破裂，強行平倉。
*   **全域斷路器 (PSL)**: 該期所有組的累計虧損達到該資金槽總資金的特定比例（如 10%）時，強制平倉所有部位並永久關閉該期剩餘的所有交易。

# 四、 主要學術文獻與理論奠基對照

本專案各策略從底層回測引擎到高維特徵分群選股，均奠基於國內外經典與前沿的學術文獻，以下進行詳細的理論奠基與專案代碼實作之對照解析：

### 1. 經典距離法的開山之作
*   **文獻**: *Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs Trading: Performance of a Relative-Value Arbitrage Rule. The Review of Financial Studies, 19(3), 797-827.*
*   **理論貢獻**: 首創對數價格 Z-Score 正規化與計算 Sum of Squared Differences (SSD) 尋找股票配對的距離套利模型。
*   **專案對照**: 為本專案的 **經典 SSD (Basic) 策略** 提供理論支撐。確立了本平台中作為基準 (Baseline) 策略的全部底層距離法篩選與信號監控架構。

### 2. 統計套利策略綜述與交易成本探討
*   **文獻**: *Krauss, C. (2015). Statistical arbitrage pairs trading strategies: Review and outlook. Journal of Economic Surveys, 29(4), 582-625. & Rad, H. et al. (2016).*
*   **理論貢獻**: 系統性對比了距離法與共整合法。深入探討了摩擦成本（手續費、滑價）對套利空間的侵蝕。
*   **專案對照**: 啟發了本專案的 **進階 SSD (OLS) 策略**，引入了交易期動態滾動回歸（OLS）估計 $\\beta$ 與雙邊交易摩擦成本的擬真回測。

### 3. 無監督聚類與降維配對選股
*   **文獻**: *Bernardo, A., & Serra, A. P. (2021). Pairs Trading via Unsupervised Learning. SSRN.*
*   **理論貢獻**: 首創將機器學習非線性流形降維 (UMAP) 與密度聚類 (HDBSCAN) 融合，打破傳統同一產業配對的僵化限制，自動過濾非平穩的特異股票。
*   **專案對照**: 本專案 **HDBSCAN (UMAP) 策略** 的核心理論靈魂，本專案完整復現了其特徵矩陣到分群排除噪聲（Label=-1）的架構。

### 4. 深度特徵工程與共整合優化 (國內學術碩士論文)
*   **文獻**: *朱羿璁 (2025). 結合共整避險比率與混合獎勵設計之強化學習配對交易模型. 國立陽明交通大學碩士論文. & 許鈞翔 (2025).*
*   **理論貢獻**: 實證了結合深度非線性自編碼器（AE）提取隱含走勢特徵、以及應用共整合比率動態對齊在亞太與全球市場實戰的優越性。
*   **專案對照**: 奠定了本專案 **HDBSCAN (AE UMAP) 策略** 的網絡架構，使用 PyTorch 實作特徵重構以濾除市場高頻隨機噪音。

### 5. 公司基本面多因子選股
*   **文獻**: *In Search of Pairs using Firm Fundamentals (2021). Journal of Financial Econometrics.*
*   **理論貢獻**: 證實了結合基本面特徵（會計特徵、行業分類、因子載荷）進行分群配對，能從源頭杜絕配對在交易期內發生結構性漂移背離的停損風險。
*   **專案對照**: 確立了本專案 **HDBSCAN MultiFactor 策略** 的因子特徵工程架構。

# 五、 經典 SSD (Basic) 策略公式與核心機制

經典最小平方距離 (SSD) 策略是無參數、純非參數統計配對的基準方法。其核心在於尋找形成期價格走勢完全平行的股票對。

### 📈 1. 形成期公式與篩選步驟

1.  **對數價格轉換與 Z-Score 標準化**:
    設第 $i$ 檔股票在形成期 $t$ 的收盤價為 $p_{i,t}$，首先將其轉換為對數價格：
    $$y_{i,t} = \\ln(p_{i,t})$$
    隨後進行 Z-Score 正規化，消除價格絕對水平與變異度的量綱影響：
    $$\\tilde{y}_{i,t} = \\frac{y_{i,t} - \\mu_i}{\\sigma_i}$$
    其中，\\(\mu_i = \\frac{1}{252} \\sum_{t=1}^{252} y_{i,t}\\) 為形成期對數價格均值，\\(\sigma_i = \\sqrt{\\frac{1}{251} \\sum_{t=1}^{252} (y_{i,t} - \\mu_i)^2}\\) 為標準差。

2.  **計算平方距離和 (SSD)**:
    對同產業中任意兩檔股票 $i$ 與 $j$，計算其標準化對數價格的平方距離和 (SSD)：
    $$SSD_{i,j} = \\sum_{t=1}^{252} (\\tilde{y}_{i,t} - \\tilde{y}_{j,t})^2$$
    挑選 $SSD_{i,j}$ 最小的前 $N$ 組配對。設 $Y = Ticker\\_A$ 為因變數，$X = Ticker\\_B$ 為自變數。避險比例固定為市場中性：$\\beta_{form} = 1.0$。

3.  **計算歷史價差統計量**:
    $$Spread_t = \\tilde{y}_{Y,t} - \\tilde{y}_{X,t}$$
    並計算形成期價差的歷史均值 $\\mu_{spread}$ 與歷史標準差 $\\sigma_{spread}$。

### 📊 2. 交易期每日信號監控

在交易期每日 $t$，套用形成期統計參數外推計算 Z-Score：
$$Z_t = \\frac{(\\tilde{y}_{Y,t} - \\tilde{y}_{X,t}) - \\mu_{spread}}{\\sigma_{spread}}$$

*   **開倉做空價差 (Short Spread)**: 當 $Z_t > 2.0$ 時進場。做空 A，做多 B。名目配置比例為 $1.0 : 1.0$。
*   **開倉做多價差 (Long Spread)**: 當 $Z_t < -2.0$ 時進場。做多 A，做空 B。名目配置比例為 $1.0 : 1.0$。
*   **平倉信號**: 當 $Z_t$ 回歸零軸時平倉。

# 六、 進階 SSD (OLS) 殘差滾動策略公式與核心機制

進階 SSD (OLS) 策略在形成期沿用距離法篩選配對，但在交易期引入了**每日滾動 OLS 回歸參數估計**，使策略具備適應中短期趨勢漂移的彈性。

### 🔄 1. 交易期滾動 OLS 參數動態估計

在交易期每日 $t$，取過去 $W$ 天（預設滾動窗口 $W = 20$ 天）的歷史標準化價格序列 $\\tilde{y}_{Y,\\tau}$ 與 $\\tilde{y}_{X,\\tau}$，建立如下動態回歸方程：
$$\\tilde{y}_{Y,\\tau} = \\alpha_t + \\beta_t \\tilde{y}_{X,\\tau} + \\epsilon_{\\tau}, \\quad \\tau \in [t-W+1, t]$$

使用普通最小二乘法 (OLS) 每日滾動求解最新的避險比例 $\\beta_t$、漂移常數項 $\\alpha_t$：
$$\\beta_t = \\frac{Cov(\\tilde{y}_{Y}, \\tilde{y}_{X})}{Var(\\tilde{y}_{X})} = \\frac{\\sum_{\\tau=t-W+1}^{t} (\\tilde{y}_{Y,\\tau} - Mean(\\tilde{y}_{Y}))(\\tilde{y}_{X,\\tau} - Mean(\\tilde{y}_{X}))}{\\sum_{\\tau=t-W+1}^{t} (\\tilde{y}_{X,\\tau} - Mean(\\tilde{y}_{X}))^2}$$
$$\\alpha_t = Mean(\\tilde{y}_{Y}) - \\beta_t Mean(\\tilde{y}_{X})$$
其中 $Mean$ 與 $Cov$ 皆基於 $W$ 天滾動窗口計算。

### 📊 2. 滾動價差與動態 Z-Score

1.  **計算當日滾動價差 (OLS Spread)**:
    $$Spread_t = \\tilde{y}_{Y,t} - \\alpha_t - \\beta_t \\tilde{y}_{X,t}$$
2.  **計算滾動殘差標準差 $\\sigma_t$**:
    $$\\sigma_t = \\sqrt{\\frac{1}{W-2} \\sum_{\\tau=t-W+1}^{t} (\\tilde{y}_{Y,\\tau} - \\alpha_t - \\beta_t \\tilde{y}_{X,\\tau})^2}$$
3.  **計算當日動態 Z-Score**:
    $$Z_t = \\frac{Spread_t}{\\sigma_t}$$

### ⚖️ 3. 部位下單與信號執行

*   **開倉做空價差**: 當 $Z_t > 2.0$ 時進場。做空 A，做多 B。部位名目配置比例為 $1.0 : |\\beta_t|$。
*   **開倉做多價差**: 當 $Z_t < -2.0$ 時進場。做多 A，做空 B。部位名目配置比例為 $1.0 : |\\beta_t|$。
*   **平倉信號**: 當 $Z_t$ 回歸零軸時平倉。部位下單股數隨當天最新 $\\beta_t$ 動態對齊以維持中性。

# 七、 HDBSCAN (UMAP) 密度分群策略公式與核心機制

該策略結合了**量化特徵工程**、**無監督降維聚類**與**嚴謹的 Engle-Granger 共整合過濾**，從源頭篩選具備長期定態均值復歸關係的黃金配對。

### 🧠 1. 多維特徵工程與標準化
在形成期，為每支股票萃取 $13$ 維特徵向量 $\\mathbf{f}_i = [f_{i,1}, \\dots, f_{i,13}]^T$。對全市場股票特徵進行 Z-Score 標準化：
$$\\mathbf{x}_i = \\frac{\\mathbf{f}_i - \\boldsymbol{\\mu}_f}{\\boldsymbol{\\sigma}_f}$$
其中 $\\boldsymbol{\\mu}_f$ 與 $\\boldsymbol{\\sigma}_f$ 分別為全市場股票特徵向量 the 均值與標準差向量。

### 📉 2. UMAP 降維與 HDBSCAN 分群
1.  **UMAP 流形降維**: 利用非線性局部流形投影，將 $\\mathbf{x}_i \\in \\mathbb{R}^{13}$ 壓縮至低維嵌入空間 $\\mathbf{z}_i \\in \\mathbb{R}^5$：
    $$\\mathbf{z}_i = \\text{UMAP}(\\mathbf{x}_i)$$
2.  **HDBSCAN 密度聚類**: 運行聚類算法（Min Cluster Size=3），獲得每支股票的群落標籤 $C_i$。剔除噪聲點（$C_i = -1$）。

### ⚖️ 3. 同產業同群落 Engle-Granger 共整合過濾
對於滿足 $\\text{Sector}_i = \\text{Sector}_j$ 且 $C_i = C_j \\neq -1$ 的股票對，在形成期內進行共整合 OLS 回歸：
$$\\ln(p_{A,\\tau}) = \\alpha + \\beta \\ln(p_{B,\\tau}) + \\epsilon_{\\tau}, \\quad \\tau \in [1, 252]$$
對殘差 $\\epsilon_{\\tau}$ 進行無常數項的 ADF 定態檢定：
$$\\Delta \\epsilon_{\\tau} = \\gamma \\epsilon_{\\tau-1} + \\sum_{k=1}^{p} \\theta_k \\Delta \\epsilon_{\\tau-k} + u_{\\tau}$$
檢定假設 $H_0: \\gamma = 0$。若 ADF $p$-value $< 0.05$，則該配對具備共整合關係。隨後計算 Ornstein-Uhlenbeck 均值復歸半衰期：
$$Half\\text{-}life = -\\frac{\\ln(2)}{\\gamma}$$
僅保留 $Half\\text{-}life \\in [2, 60]$ 天的配對，並依 ADF 統計量升序挑選 $Top\\_N$。在交易期中，以 OLS 殘差 Z-Score 執行交易。

# 八、 HDBSCAN (AE UMAP) 深度學習策略公式與核心機制

HDBSCAN (AE UMAP) 策略在無監督分群的特徵提取階段，引進了 **深度神經自編碼器 (Autoencoder, AE)**，透過非線性重構以濾除高頻市場雜訊。

### 🧬 1. PyTorch 自編碼器網絡公式與訓練

在形成期，將股票的標準化 $13$ 維特徵 $\\mathbf{x} \\in \\mathbb{R}^{13}$ 輸入自編碼器網絡：

1.  **編碼器 (Encoder) 前向傳播**:
    $$\\mathbf{h}_1 = \\text{ReLU}(\\mathbf{W}_1 \\mathbf{x} + \\mathbf{b}_1), \\quad \\mathbf{W}_1 \\in \\mathbb{R}^{8 \times 13}$$
    $$\\mathbf{z}_{latent} = \\text{ReLU}(\\mathbf{W}_2 \\mathbf{h}_1 + \\mathbf{b}_2), \\quad \\mathbf{W}_2 \\in \\mathbb{R}^{5 \times 8}$$
    其中瓶頸層（Bottleneck Layer）輸出的 $\\mathbf{z}_{latent} \\in \\mathbb{R}^5$ 即為過濾噪聲後的低維深度嵌入特徵。這裡的 $\\mathbf{b}_2$ 與各偏置項均進行了雙反斜線轉義。

2.  **解碼器 (Decoder) 前向重構**:
    $$\\mathbf{h}_2 = \\text{ReLU}(\\mathbf{W}_3 \\mathbf{z}_{latent} + \\mathbf{b}_3), \\quad \\mathbf{W}_3 \\in \\mathbb{R}^{8 \times 5}$$
    $$\\mathbf{\\hat{x}} = \\mathbf{W}_4 \\mathbf{h}_2 + \\mathbf{b}_4, \\quad \\mathbf{W}_4 \\in \\mathbb{R}^{13 \times 8}$$
    其中 $\\mathbf{\\hat{x}} \\in \\mathbb{R}^{13}$ 為特徵的重構輸出。

3.  **損失函數與訓練**:
    使用均方誤差 (MSE) 作為重構損失：
    $$\\mathcal{L}_{MSE} = \\frac{1}{M} \\sum_{i=1}^{M} \\|\\mathbf{x}_i - \\mathbf{\\hat{x}}_i\\|^2$$
    利用 Adam 優化器訓練 50 個 Epoch 以極小化重構誤差。訓練完成後，提取每支股票的非線性特徵嵌入 $\\mathbf{z}_{latent}$。

### 📉 2. 聚類、協整與信號執行

將提取的 $\\mathbf{z}_{latent}$ 輸入 UMAP 進行降維，並使用 HDBSCAN 密度聚類分群。在同聚類群落且同行業內進行 Engle-Granger 二階段共整合過濾與半衰期篩選（同 UMAP 策略），選出最佳 $Top\\_N$ 配對並在交易期中外推交易。

# 九、 HDBSCAN MultiFactor 多因子策略公式與核心機制

多因子 HDBSCAN 策略在量價統計特徵的基礎上，融合了**股票基本面因子**與**產業因子**，以實現全方位的特徵匹配，從源頭規避交易期基本面漂移導致的停損風險。

### 📊 1. 多因子特徵融合矩陣

設每支股票除了 $13$ 維的量價統計特徵 $\\mathbf{x}_i$ 外，還包含基本面因子特徵向量 $\\mathbf{g}_i = [g_{i,1}, \\dots, g_{i,K}]^T$（包含市值對數、估值特徵、及行業分類對照表 `imputed_sectors.csv` 的 One-Hot 編碼特徵向量等）。

我們建立融合特徵矩陣 $\\mathbf{F}_i \\in \\mathbb{R}^{13 + K}$：
$$\\mathbf{F}_i = [\\mathbf{x}_i, \\mathbf{g}_i]^T$$

將 $\\mathbf{F}_i$ 在形成期進行 Z-Score 標準化，作為聚類的特徵基礎。

### 📉 2. 降維、聚類與 Engle-Granger 共整合

1.  **流形降維**: 利用 UMAP 將多因子特徵 $\\mathbf{F}_i$ 降維至 5 維嵌入空間。
2.  **HDBSCAN 聚類**: 在多因子降維空間中運行 HDBSCAN 密度分群，自動尋找基本面與量價走勢高度共鳴的股票群落。
3.  **EG 共整合與半衰期過濾**: 對同群落同產業內的股票進行協整 OLS 回歸與 ADF 定態檢定，估計 $\\beta$、\\(\\alpha\\) 與殘差半衰期：
    $$\\ln(p_{A,\\tau}) = \\alpha + \\beta \\ln(p_{B,\\tau}) + \\epsilon_{\\tau}$$
    篩選 ADF $p$-value $< 0.05$ 且 $Half\\text{-}life \\in [2, 60]$ 天的配對，按 ADF 顯著度升序選取 $Top\\_N$。在交易期中每日計算 OLS 殘差 Z-Score 執行部位等權重配置與多層級風控。

# 十、 各策略差異多維度對比矩陣

不同配對交易策略在模型複雜度、統計假設、適應能力與計算效率上存在顯著的差異。以下針對五大策略進行系統性的多維度比對與優缺點剖析。

<table style="width: 100%; border-collapse: collapse; font-family: 'Inter', 'Outfit', sans-serif; font-size: 0.50em; margin: 10px auto; text-align: center; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.08);">
  <thead>
    <tr style="background-color: #2D3748; color: #ffffff; font-weight: 600; text-transform: uppercase;">
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">策略名稱</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">特徵提取與降維</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">配對篩選機制</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">參數更新頻率</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">主要核心優勢</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">主要面臨挑戰</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #ffffff;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold;">經典 SSD (Basic)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Z-Score 價格標準化 / 無降維</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">最小歐氏平方距離平方和 (SSD)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">形成期固定，交易期不更新</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">計算極速，模型無超參數，極度穩健且不易過度擬合</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">無法捕捉非線性相關；交易期若參數漂移會產生重大虧損</td>
    </tr>
    <tr style="background-color: #f7fafc;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold;">進階 SSD (OLS)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">價格標準化 / OLS 滾動回歸</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">SSD 挑選配對 + OLS 參數估計</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">交易期滾動更新 (如 20 天視窗)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">動態更新 $\\beta$ 與 $\\sigma$，具有極強的短期市場適應力</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">頻繁計算 OLS 增加摩擦成本，窗口過短易造成部位信號抖動</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold;">HDBSCAN (UMAP)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">13維量價特徵 / UMAP 非線性降維</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">HDBSCAN 密度分群 + EG 共整合檢定 (ADF p < 0.05)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">形成期固定 / 交易期動態 Z-Score</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">無監督聚類自動排除噪聲股，共整合保證價差長期回歸</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">UMAP 具有隨機性且對超參數敏感；共整合檢定運算量較大</td>
    </tr>
    <tr style="background-color: #f7fafc;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold;">HDBSCAN (AE UMAP)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Autoencoder 深度非線性壓縮 / UMAP</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">深度特徵聚類 + EG 共整合雙重篩選</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">形成期重訓 AE 網路 / 交易期外推</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">自編碼器強大重構能力，能極佳地濾除價格隨機噪聲</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">模型參數極多，黑盒特徵難以直觀解釋，硬體計算要求高</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold;">HDBSCAN MultiFactor</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">基本面特徵對照表 + 統計因子特徵</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">基本面與量價多維度密度分群 + EG 共整合</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">形成期固定 / 交易期執行</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">從源頭限縮同產業與同基本面特徵，配對基本面背離率最低</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; text-align: left;">高度依賴外部基本面數據完整度，特徵尺度縮放需精細調校</td>
    </tr>
  </tbody>
</table>

# 十一、 回測績效比較與走勢圖

本章節展示各策略在最優參數組合下的實時回測績效指標，並以 Plotly 互動式折線圖動態呈現各策略累計帳戶淨值的走勢對比。

## 📈 六大策略最優參數回測效能對比 (實時更新)

<table style="width: 100%; border-collapse: collapse; font-family: 'Inter', 'Outfit', sans-serif; font-size: 0.52em; margin: 10px auto; text-align: center; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.08);">
  <thead>
    <tr style="background-color: #1a365d; color: #ffffff; font-weight: 600; text-transform: uppercase;">
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">策略名稱 (Method)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">最佳參數組合 (Optimal Params)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">最終淨值 (Final Equity)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">年化報酬 (Ann. Return)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">最大回撤 (Max DD)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">夏普值 (Sharpe)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">RCC (%)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">REC (%)</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #ffffff;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; ">經典 SSD (Basic) (CURRENT)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 5, SL: 0%, ZWin: 0, PSL: 無, MSR: 30%, DSZ: 無, VolAdj: 無</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$10,748.71</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+0.28%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-10.15%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">0.14</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+7.49%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+0.05%</td>
    </tr>
    <tr style="background-color: #f0f7ff; font-weight: bold; border: 2px solid #3182ce;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; color: #2b6cb0;">進階 SSD (OLS) 🌟 (CURRENT)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 10, SL: 0%, ZWin: 0, PSL: 無, MSR: 無, DSZ: 無, VolAdj: 無</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$12,000.72</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+0.73%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-17.16%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">0.28</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+20.01%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+0.25%</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; ">HDBSCAN (UMAP) (CURRENT)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 10, SL: 15%, ZWin: 0, PSL: 無, MSR: 無, DSZ: 無, VolAdj: 無</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$8,467.13</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.66%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-25.03%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-0.19</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-15.33%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.08%</td>
    </tr>
    <tr style="background-color: #f8fafc;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; ">HDBSCAN (MF) (CURRENT)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 5, SL: 15%, ZWin: 0, PSL: 無, MSR: 無, DSZ: 無, VolAdj: 無</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$10,430.13</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+0.18%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-19.52%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">0.04</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+4.30%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #38a169;">+0.02%</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; ">HDBSCAN (PCA) (CURRENT)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 20, SL: 0%, ZWin: 0, PSL: 無, MSR: 30%, DSZ: 無, VolAdj: 無</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$8,575.36</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.61%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-25.09%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-0.23</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-14.25%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.09%</td>
    </tr>
    <tr style="background-color: #f8fafc;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; ">HDBSCAN (UMAP) (FULL)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 10, SL: 5%, ZWin: 0, PSL: 無, MSR: 無, DSZ: 無, VolAdj: 無</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$9,608.54</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.16%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-20.96%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-0.05</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-3.91%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.02%</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; ">HDBSCAN (MF) (FULL)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 5, SL: 0%, ZWin: 0, PSL: 無, MSR: 30%, DSZ: 無, VolAdj: 無</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$9,974.65</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.02%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-14.08%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-0.00</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.25%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.00%</td>
    </tr>
    <tr style="background-color: #f8fafc;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; ">HDBSCAN (PCA) (FULL)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 20, SL: 0%, ZWin: 0, PSL: 無, MSR: 30%, DSZ: 無, VolAdj: 無</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$8,391.42</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.69%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-24.67%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-0.26</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-16.09%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.09%</td>
    </tr>
  </tbody>
</table>

> [!NOTE]
> **RCC (Return on Capital Constraint)**: 基於回測期分配總資金 $10,000 計算。
> **REC (Return on Engaged Capital)**: 基於實際動用且對齊 Beta 避險權重的保證金資金計算。
> *資料更新時間: 當前電腦編譯實時生成。



## 11.2 策略累計權益曲線對比

本專案使用高級的互動式雙通道 Plotly 繪圖模組，支援一鍵切換**「當前資料集 (Current)」**與**「完整歷史資料集 (Full History)」**。以下透過 RevealJS iframe 元件完美展示這個可交互、防重疊的淨值走勢圖：

<iframe src="iframe_figures/figure_4.html" width="100%" height="650px" style="border:none; background:white; border-radius:8px; box-shadow: 0 4px 12px rgba(0,0,0,0.08);"></iframe>